In [1]:
import efficient_fpt
from efficient_fpt.multi_stage_cy import compute_loss_parallel, print_num_threads
import hssm
import matplotlib.pyplot as plt 

In [2]:
print_num_threads()

Number of available threads: 48


In [3]:
import pandas as pd
import numpy as np 
import pickle as pkl 

data = pkl.load(open("/users/azhan378/data/azhang/HSSM/addm_andrew_dev /addm_data_20251015-163921.pkl", "rb"))

data_rt_choice = pd.DataFrame({
    'rt': data['decision_data'][:1000,0].astype(np.float64),
    'response': data['decision_data'][:1000,1].astype(np.int32)
})
# data_rt_choice = np.column_stack((
#     data['decision_data'][:1000, 0].astype(np.float64),
#     data['decision_data'][:1000, 1].astype(np.int32)
# ))
data_rt_choice['rt']

0      3.508225
1      1.070135
2      2.813685
3      3.210385
4      2.170045
         ...   
995    2.379745
996    2.530485
997    1.068735
998    0.650125
999    0.715705
Name: rt, Length: 1000, dtype: float64

In [4]:
# smaller subset 
DATA_TYPE = np.float64

a_small = data["a"]
b_small = data["b"]
x0_small = data["x0"]
eta_small = data["eta"]
kappa_small = data["kappa"]
r1_data_small = data["r1_data"][:1000]
r2_data_small = data["r2_data"][:1000]
flag_data_small = data["flag_data"][:1000].astype(np.int32)
sigma_small = data["sigma"]
T_small = data["T"]
mu1_true_data_small = kappa_small * (r1_data_small - eta_small * r2_data_small)
mu2_true_data_small = kappa_small * (eta_small * r1_data_small - r2_data_small)
mu_true_data_small = data["mu_array_padded_data"][:1000].astype(DATA_TYPE)
sacc_data_small = data["sacc_array_padded_data"][:1000].astype(DATA_TYPE)
length_data_small = data["d_data"][:1000].astype(np.int32)

num_data_small, max_d_small = mu_true_data_small.shape
print(num_data_small, max_d_small)

1000 12


In [5]:
print(np.asarray(a_small).shape)

()


In [6]:
loglik = compute_loss_parallel(
       mu1_true_data_small,
       mu2_true_data_small,
       np.array(data_rt_choice['rt'], dtype=DATA_TYPE), 
       np.array(data_rt_choice['response'], dtype=np.int32),
       flag_data_small,
       sacc_data_small,
       length_data_small,
       max_d_small,
       sigma_small,
       a_small,
       b_small,
       x0_small,
       num_threads = 32
    )
print("Loglikelihood:", loglik)
total_loglik = loglik*-len(data_rt_choice['rt'])
print(total_loglik)

Loglikelihood: 1.4938145464848105
-1493.8145464848105


In [ ]:
# blackbox likelihood function 

def addm_loglikelihood(data_model, a, b, x0, eta, kappa): 
   # Extract scalar values from parameter arrays -- otherwise can't convert multi-d array to scalar 
   a = np.atleast_1d(a)[0] if np.ndim(a) > 0 else a
   b = np.atleast_1d(b)[0] if np.ndim(b) > 0 else b
   x0 = np.atleast_1d(x0)[0] if np.ndim(x0) > 0 else x0
   eta = np.atleast_1d(eta)[0] if np.ndim(eta) > 0 else eta
   kappa = np.atleast_1d(kappa)[0] if np.ndim(kappa) > 0 else kappa
   
   # Handle data_model as either DataFrame or numpy array
   data_lik = np.asarray(data_model)
   if data_lik.ndim == 1:
       # If 1D, reshape to 2D
       data_lik = data_lik.reshape(-1, 2)
   
   # Get number of observations from data shape
   n_obs = data_lik.shape[0]
   
   r1_data = r1_data_small
   r2_data = r2_data_small
   flag_data = flag_data_small
   sigma = sigma_small
#  T = T_small
#  # mu_array_padded = mu_true_data_small
   sacc_array_padded = sacc_data_small
   length_data = length_data_small

   mu1_true_data = kappa * (r1_data - eta * r2_data)
   mu2_true_data = kappa * (eta * r1_data - r2_data)

   loglik = compute_loss_parallel(
      mu1_true_data,
      mu2_true_data,
      data_lik[:,0].astype(DATA_TYPE),
      data_lik[:,1].astype(np.int32),
      flag_data,
      sacc_array_padded,
      length_data,
      max_d_small,
      sigma,
      a,
      b,
      x0,
      num_threads = 32
   )
   loglik_convert = -n_obs * loglik
   print("this is the loglik", loglik_convert)
   return loglik_convert

In [11]:
# Blackbox custom hssm model 

# Define priors for parameters
M = max_d_small  # Maximum decision time for boundary constraint

addm_model_small = hssm.HSSM(
    data=data_rt_choice, 
    model="addm", 
    model_config = {
        "list_params": ["a", "b", "x0", "eta", "kappa"],
        "bounds": {
            "a": (0.5, 3.0),
            "b": (0.5, 3.0),
            "x0": (-0.5, 0.5),
            "eta": (0.01, 0.5),
            "kappa": (0.01, 1.0)
        },
        'choices': [-1,1]
    },
    loglik=addm_loglikelihood,
    loglik_kind="blackbox",
    include=[
        {"name": "eta", "prior": {"name": "Beta", "alpha": 2.0, "beta": 2.0}},
        {"name": "kappa", "prior": {"name": "Gamma", "alpha": 2.0, "beta": 4.0}},
        {"name": "a", "prior": {"name": "Gamma", "alpha": 4.0, "beta": 2.0}},
        {"name": "b", "prior": {"name": "Beta", "alpha": 2.0, "beta": 2.0}},
        {"name": "x0", "prior": {"name": "Beta", "alpha": 2.0, "beta": 2.0}},
    ],
)

You supplied a model 'addm', which is currently not supported in the ssm_simulators package. An error will be thrown when sampling from the random variable or when using any posterior or prior predictive sampling methods.


Model initialized successfully.


In [12]:
sample = addm_model_small.sample(chains=1, draws=100, tune=100)

Using default initvals. 



Sequential sampling (1 chains in 1 job)
CompoundStep
>Slice: [kappa]
>Slice: [eta]
>Slice: [x0]
>Slice: [b]
>Slice: [a]


Output()

this is the loglik -1823.4531225755202

AssertionError: SpecifyShape: Got 0 dimensions, expected 1 dimensions.
Apply node that caused the error: SpecifyShape(BlackBoxOp.0, 1000)
Toposort index: 12
Inputs types: [TensorType(float64, shape=(None,)), TensorType(int16, shape=())]
Inputs shapes: [(), ()]
Inputs strides: [(), ()]
Inputs values: [array(-1823.45312258), array(1000, dtype=int16)]
Outputs clients: [[Composite{log((0.0025000000000000005 + (0.95 * exp(i0))))}(SpecifyShape.0)]]

HINT: Re-running with most PyTensor optimizations disabled could provide a back-trace showing when this node was created. This can be done by setting the PyTensor flag 'optimizer=fast_compile'. If that does not work, PyTensor optimizations can be disabled with 'optimizer=None'.
HINT: Use the PyTensor flag `exception_verbosity=high` for a debug print-out and storage map footprint of this Apply node.